# peers

> The peers folder: the envelope a message travels in, the log of all messages, and one inbox per session.

In [0]:
#| default_exp peers

In [0]:
#| hide
from nbdev.showdoc import *

In [0]:
#| hide
from fastcore.test import test_eq

Claude Code, Codex and Ramabana sessions on one Mac can send each other notes, asks and replies. This module keeps all of it on disk, in one peers folder. It needs no network and no model, so a hook can import it and stay fast.

A message is an `Envelope`. `post` writes it to the receiver's inbox file and to the log. `take` reads the inbox and marks what it read, so two readers never get the same message. `control_call` is the old way to reach a running session; it lives here so this module needs nothing from `control`.

In [0]:
#| export
import fcntl, json, os, socket, time, uuid
from dataclasses import dataclass, field, asdict, fields
from pathlib import Path

## The folder

`peers_dir` gives the peers folder. `$RAMABANA_PEERS` wins, so tests and scripts can point it elsewhere. A peer key is `f'{kind}-{session}'`; it names the peer's inbox.

In [0]:
#| export
def peers_dir(cfg=None):
    "The peers folder: `$RAMABANA_PEERS`, else `peers` in `cfg` (default `~/.config/ramabana`); made on first use."
    d = Path(os.environ.get('RAMABANA_PEERS') or Path(cfg or '~/.config/ramabana').expanduser()/'peers')
    d.mkdir(parents=True, exist_ok=True)
    return d

In [0]:
import tempfile
with tempfile.TemporaryDirectory() as t:
    test_eq(peers_dir(t), Path(t)/'peers')
    assert (Path(t)/'peers').is_dir()

## The envelope

An `Envelope` is one message. `kind` is `note`, `ask` or `reply`. `frm` says who sent it and must hold a `key` (`f'{kind}-{session}'`), because later code counts budgets by that key. `id`, `thread` and `at` fill themselves in. In `dict()` the field `frm` is written as `from`, which is a Python keyword.

In [0]:
#| export
@dataclass
class Envelope:
    "One message between peers."
    kind: str
    text: str
    frm: dict
    to: str = ''
    repo: str = ''
    thread: str = ''
    hops: int = 0
    changes: list = field(default_factory=list)
    proof: list = field(default_factory=list)
    id: str = ''
    at: float = 0.
    def __post_init__(self):
        self.id = self.id or uuid.uuid4().hex[:12]
        self.thread = self.thread or self.id
        self.at = self.at or time.time()
    def dict(self):
        "The envelope as a dict, with `frm` written as `from`."
        d = asdict(self)
        d['from'] = d.pop('frm')
        return d
    @classmethod
    def load(cls, d):
        "The envelope in `d`, a dict made by `dict`; keys that are not fields are ignored."
        d = dict(d, frm=d.get('from', d.get('frm', {})))
        return cls(**{f.name: d[f.name] for f in fields(cls) if f.name in d})

In [0]:
e = Envelope('ask', 'build Card', {'kind': 'claude', 'session': 's1', 'key': 'claude-s1'}, to='leela')
test_eq(Envelope.load(e.dict()), e)
test_eq((e.dict()['from']['key'], e.thread == e.id, len(e.id)), ('claude-s1', True, 12))

## The log and the inboxes

`log` adds one row to `log.jsonl`; `read_log` reads the rows from a time on. A row is an envelope dict or an event row such as `{'event': 'joined', 'at': ...}`. `post` writes an envelope to the receiver's inbox and to the log.

In [0]:
#| export
def _append(p, row):
    "Add `row` as one line to `p`."
    with open(p, 'a') as f: f.write(json.dumps(row) + '\n')

def _rows(p):
    "The rows in the JSON lines file `p`; none when it does not exist."
    return [json.loads(l) for l in p.read_text().splitlines() if l.strip()] if p.exists() else []

def log(d, row):
    "Add `row` to the log in `d`."
    _append(Path(d)/'log.jsonl', row)

def read_log(d, since=0.):
    "The log rows in `d` made after time `since`."
    return [r for r in _rows(Path(d)/'log.jsonl') if r.get('at', 0) > since]

def post(d, key, env):
    "Put `env` in the inbox of peer `key` and in the log."
    _append(Path(d)/f'{key}.inbox.jsonl', env.dict())
    log(d, env.dict())

In [0]:
with tempfile.TemporaryDirectory() as t:
    log(t, {'event': 'joined', 'at': 5.})
    post(t, 'k', Envelope('note', 'hi', {'key': 'claude-s1'}, at=9.))
    test_eq([r.get('event') or r['text'] for r in read_log(t)], ['joined', 'hi'])
    test_eq(len(read_log(t, since=6)), 1)

## Taking messages

`take` is the only way to read an inbox. It locks the `.taken` file, reads the rows not yet taken that match `pred`, and writes their ids there. The lock means two readers, such as a hook and `tell --wait`, never take the same message. `wait_take` repeats `take` until a message comes, the time ends, or `alive()` is false.

In [0]:
#| export
def take(d, key, pred=None):
    "The inbox rows of `key` not yet taken that match `pred`, marked taken by id under a lock."
    d = Path(d)
    with open(d/f'{key}.taken', 'a+') as f:
        fcntl.flock(f, fcntl.LOCK_EX)
        f.seek(0)
        seen = set(f.read().split())
        new = [e for e in map(Envelope.load, _rows(d/f'{key}.inbox.jsonl')) if e.id not in seen and (pred is None or pred(e))]
        f.write(''.join(f'{e.id}\n' for e in new))
    return new

def wait_take(d, key, timeout, pred=None, alive=lambda: True, poll=.5):
    "`take` until a message comes; give up after `timeout` seconds (None: never) or when `alive()` is false."
    end = None if timeout is None else time.monotonic() + timeout
    while not (got := take(d, key, pred)):
        if not alive() or (end is not None and time.monotonic() >= end): break
        time.sleep(poll)
    return got

In [0]:
with tempfile.TemporaryDirectory() as t:
    post(t, 'k', Envelope('reply', 'r', {}, thread='t1'))
    post(t, 'k', Envelope('note', 'n', {}))
    test_eq([e.text for e in take(t, 'k', lambda e: e.thread == 't1')], ['r'])
    test_eq([e.text for e in take(t, 'k')], ['n'])
    test_eq(take(t, 'k'), [])
    test_eq(wait_take(t, 'k', .1, poll=.02), [])

## The control socket call

`control_call(path, method, **params)` sends one request to a session's control socket and returns the `result`. An error reply raises `RuntimeError`. It moved here from `control`, which still exports it.

In [0]:
#| export
def control_call(path, method, timeout=30, **params):
    "Send one request to the `Control` socket at `path` and return its `result`; an error reply raises `RuntimeError`."
    with socket.socket(socket.AF_UNIX) as s:
        s.settimeout(timeout)
        s.connect(str(path))
        s.sendall((json.dumps({'id': 1, 'method': method, 'params': params}) + '\n').encode())
        out = json.loads(s.makefile('rb').readline())
    if 'error' in out: raise RuntimeError(out['error']['message'])
    return out['result']

`control_call` needs a live session, so `control` tests it with a real socket. Here a check that it exists and fails clearly when nothing listens.

In [0]:
try: control_call('/tmp/ramabana-no-such.sock', 'status', timeout=1)
except OSError as ex: print(type(ex).__name__)